# Fine-tune LLM with PyTorch FSDP and QLora on Amazon SageMaker AI using ModelTrainer

In this notebook, we fine-tune LLM on Amazon SageMaker AI, using Python scripts and the SageMaker Python SDK v3 `ModelTrainer` for executing a training job, and submit the jobs to SageMaker Training Job Batch queues.

## Prerequisites

In [1]:
# [exec-copy] pip installs disabled; using .v3test-venv local V3 SDK
print('pip install skipped in execution copy')

pip install skipped in execution copy


***

## Setup Configuration file path

In [2]:
import os
import sys

module_path = os.path.abspath(os.path.join("../.."))

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import os

model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

os.environ["model_id"] = model_id

***

## Visualize and upload the dataset

We are going to load [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset

In [4]:
from sagemaker.core.helper.session_helper import Session
import boto3
boto_session = boto3.Session(region_name="us-west-2")
sagemaker_session = Session(boto_session=boto_session)
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

[07/16/26 09:04:56] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=2925623;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=2925624;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


In [5]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

df = pd.DataFrame(dataset['train'])
df = df[:1000]

df.head()

[07/16/26 09:04:58] INFO     TensorFlow version 2.21.0 available.                                     ]8;id=2925631;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/datasets/config.py\config.py]8;;\:]8;id=2925632;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/datasets/config.py#112\112]8;;\

                    INFO     HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 ]8;id=2925639;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925640;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             200 OK"                                                                               

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925645;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925646;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoni                
                             ng-SFT/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"                       

                    WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please   ]8;id=2925653;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/huggingface_hub/utils/_http.py\_http.py]8;;\:]8;id=2925654;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/huggingface_hub/utils/_http.py#951\951]8;;\
                             set a HF_TOKEN to enable higher rate limits and faster downloads.                     

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925659;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925660;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/datasets/FreedomIntelligence/                
                             medical-o1-reasoning-SFT/fc2c9e8a37b38f38da6d449564a8c350b244aef4/READ                
                             ME.md "HTTP/1.1 200 OK"                                                               

...

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925713;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925714;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoni                
                             ng-SFT/resolve/fc2c9e8a37b38f38da6d449564a8c350b244aef4/medical_o1_sft                
                             .json "HTTP/1.1 302 Found"                                                            

medical_o1_sft.json: reconstructing file:   0%|          |  0.00B / 58.2MB            

medical_o1_sft.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

,Question,Complex_CoT,Response
0,Given the symptoms of sudden weakness in the l...,"Okay, let's see what's going on here. We've go...",The specific cardiac abnormality most likely t...
1,A 33-year-old woman is brought to the emergenc...,"Okay, let's figure out what's going on here. A...","In this scenario, the most likely anatomical s..."
2,A 61-year-old woman with a long history of inv...,"Okay, let's think about this step by step. The...",Cystometry in this case of stress urinary inco...
3,A 45-year-old man with a history of alcohol us...,"Alright, let’s break this down. We have a 45-y...",Considering the clinical presentation of sudde...
4,A 45-year-old man presents with symptoms inclu...,"Okay, so here's a 45-year-old guy who's experi...",Based on the clinical findings presented—wide-...


In [6]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.1, random_state=42)

print("Number of train elements: ", len(train))
print("Number of test elements: ", len(val))

Number of train elements:  900
Number of test elements:  100


Create a prompt template and load the dataset with a random sample to try summarization.

In [7]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(model_id)

def prepare_dataset(sample):
    system_text = (
        "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.\n"
        "Below is an instruction that describes a task, paired with an input that provides further context.\n"
        "Write a response that appropriately completes the request.\n"
        "Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response."
    )

    messages = []
    messages.append({"role": "system", "content": system_text})
    messages.append({"role": "user", "content": sample["Question"]})

    # Use different tags that won't be detected by the template
    messages.append(
        {
            "role": "assistant",
            "content": f"\n[REASONING_START]\n{sample['Complex_CoT']}\n[REASONING_END]\n{sample['Response']}",
        }
    )

    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)

    # Replace with actual think tags after template processing
    formatted_text = formatted_text.replace("[REASONING_START]", "<think>")
    formatted_text = formatted_text.replace("[REASONING_END]", "</think>")

    sample["text"] = formatted_text

    return sample

[07/16/26 09:05:12] INFO     HTTP Request: HEAD                                                     ]8;id=2925719;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925720;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolv                
                             e/main/config.json "HTTP/1.1 307 Temporary Redirect"                                  

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925725;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925726;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/deepseek-ai/DeepSeek-R                
                             1-Distill-Llama-8B/6a6f4aa4197940add57724a7707d069478df56b1/config.jso                
                             n "HTTP/1.1 200 OK"                                                                   

                    INFO     HTTP Request: GET                                                      ]8;id=2925731;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925732;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/deepseek-ai/DeepSeek-R                
                             1-Distill-Llama-8B/6a6f4aa4197940add57724a7707d069478df56b1/config.jso                
                             n "HTTP/1.1 200 OK"                                                                   

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925737;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925738;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolv                
                             e/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"                        

...

[07/16/26 09:05:13] INFO     HTTP Request: HEAD                                                     ]8;id=2925797;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925798;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolv                
                             e/main/tokenizer.model "HTTP/1.1 404 Not Found"                                       

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925803;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925804;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolv                
                             e/main/added_tokens.json "HTTP/1.1 404 Not Found"                                     

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925809;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925810;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolv                
                             e/main/special_tokens_map.json "HTTP/1.1 404 Not Found"                               

                    INFO     HTTP Request: HEAD                                                     ]8;id=2925815;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925816;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolv                
                             e/main/chat_template.jinja "HTTP/1.1 404 Not Found"                                   

[07/16/26 09:05:14] INFO     HTTP Request: GET                                                      ]8;id=2925821;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2925822;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/deepseek-ai/DeepSeek-R1-Distill-Llam                
                             a-8B "HTTP/1.1 200 OK"                                                                

Use the Hugging Face Trainer class to fine-tune the model. Define the hyperparameters we want to use. We also create a DataCollator that will take care of padding our inputs and labels.

In [8]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)

dataset = DatasetDict({"train": train_dataset, "val": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset, remove_columns=list(train_dataset.features)
)

print(train_dataset[randint(0, len(dataset))]["text"])

val_dataset = dataset["val"].map(
    prepare_dataset, remove_columns=list(val_dataset.features)
)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

<｜begin▁of▁sentence｜>You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.<｜User｜>If a contraceptive has a failure rate of 15, how many unplanned pregnancies is a female expected to have during her reproductive period?<｜Assistant｜>
<think>
Alright, so I've got a contraceptive failure rate of 15 to think about. This means there's a 15% chance of ending up with an unplanned pregnancy each year while using this method. Okay, that's pretty clear.

Now, let's consider the reproductive period of a woman. Usually, it starts around age 15 and goes till about age 45. That's about 30 years in total. So, I'm looking at a 30-year window where this ris

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### Upload to Amazon S3

In [9]:
import boto3
import shutil

In [10]:
s3_client = boto3.client('s3', region_name="us-west-2")

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=2925827;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=2925828;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [11]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-estimator-sft-batch"
else:
    input_path = f"datasets/llm-fine-tuning-estimator-sft-batch"

# Save datasets to s3
# We will fine tune only with 20 records due to limited compute resource for the workshop
train_dataset.to_json("./data/train/dataset.json", orient="records")
val_dataset.to_json("./data/val/dataset.json", orient="records")

s3_client.upload_file(
    "./data/train/dataset.json", bucket_name, f"{input_path}/train/dataset.json"
)
train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.json"
s3_client.upload_file(
    "./data/val/dataset.json", bucket_name, f"{input_path}/val/dataset.json"
)
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.json"

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training data uploaded to:
s3://sagemaker-us-west-2-729646638167/datasets/llm-fine-tuning-estimator-sft-batch/train/dataset.json
s3://sagemaker-us-west-2-729646638167/datasets/llm-fine-tuning-estimator-sft-batch/val/dataset.json


***

## Model fine-tuning

We are now ready to fine-tune our model. We will use the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) from transfomers to fine-tune our model. We prepared a script [train.py](./scripts/train.py) which will loads the dataset from disk, prepare the model, tokenizer and start the training.

For configuration we use `TrlParser`, that allows us to provide hyperparameters in a `yaml` file. This yaml will be uploaded and provided to Amazon SageMaker similar to our datasets. Below is the config file for fine-tuning the model on `ml.g5.12xlarge`. We are saving the config file as `args.yaml` and upload it to S3.

In [12]:
%%bash

cat > ./args.yaml <<EOF
model_id: "${model_id}"       # Hugging Face model id
# sagemaker specific parameters
output_dir: "/opt/ml/model"                       # path to where SageMaker will upload the model 
train_dataset_path: "/opt/ml/input/data/train/"   # path to where FSx saves train dataset
test_dataset_path: "/opt/ml/input/data/val/"     # path to where FSx saves test dataset
# training parameters
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1                 
learning_rate: 2e-4                    # learning rate scheduler
num_train_epochs: 1                    # number of training epochs
per_device_train_batch_size: 2         # batch size per device during training
per_device_eval_batch_size: 1          # batch size for evaluation
gradient_accumulation_steps: 2         # number of steps before performing a backward/update pass
gradient_checkpointing: true           # use gradient checkpointing
bf16: true                             # use bfloat16 precision
tf32: false                            # use tf32 precision
fsdp: "full_shard auto_wrap offload"
fsdp_config: 
    backward_prefetch: "backward_pre"
    cpu_ram_efficient_loading: true
    offload_params: true
    forward_prefetch: false
    use_orig_params: true
merge_weights: true                    # merge weights in the base model
EOF

Lets upload the config file to S3.

In [13]:
import os
from sagemaker.core.s3 import S3Uploader

if default_prefix:
    input_path = f"s3://{bucket_name}/{default_prefix}/datasets/llm-fine-tuning-estimator-sft-batch"
else:
    input_path = f"s3://{bucket_name}/datasets/llm-fine-tuning-estimator-sft-batch"

# upload the model yaml file to s3
model_yaml = "args.yaml"
train_config_s3_path = S3Uploader.upload(
    local_path=model_yaml, desired_s3_uri=f"{input_path}/config"
)

os.remove("./args.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

Training config uploaded to:
s3://sagemaker-us-west-2-729646638167/datasets/llm-fine-tuning-estimator-sft-batch/config/args.yaml


## Create the ModelTrainer

Below the ModelTrainer will be used to submit the jobs

#### Get PyTorch image_uri

We are going to use the native PyTorch container image, pre-built for Amazon SageMaker

In [14]:
from sagemaker.core import image_uris
from sagemaker.core.config import load_sagemaker_config

[07/16/26 09:05:15] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=2925833;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=2925834;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

In [15]:
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
configs = load_sagemaker_config()

In [16]:
instance_type = "ml.g6.12xlarge" # Override the instance type if you want to get a different container version
instance_count = 1

instance_type

'ml.g6.12xlarge'

In [17]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.6",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

[07/16/26 09:05:16] INFO     Defaulting to only available Python version: py312                   ]8;id=2925841;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=2925842;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#615\615]8;;\

'763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.6-gpu-py312'

In [18]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

In [19]:
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train.distributed import Torchrun

role = "arn:aws:iam::729646638167:role/SageMakerRole"  # [exec-copy] explicit role; get_execution_role() unavailable locally

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=0
)

# define Training Job Name
job_name = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft-batch"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={
        "config": "/opt/ml/input/data/config/args.yaml"  # path to TRL config which was uploaded to s3
    },
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    role=role,
)

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=2925849;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2925850;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Role 'arn:aws:iam::729646638167:role/SageMakerRole' validated ]8;id=2925857;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=2925858;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             for training. Using it.                                                               

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=2925864;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2925865;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#204\204]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=2925872;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=2925873;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.6                     
                             -gpu-py312                                                                            

In [20]:
from sagemaker.train.configs import InputData

# Pass the input data
train_input = InputData(
    channel_name="train",
    data_source=train_dataset_s3_path,  # S3 path where training data is stored
)

val_input = InputData(
    channel_name="val",
    data_source=val_dataset_s3_path,  # S3 path where validation data is stored
)

config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path,  # S3 path where training config is stored
)

# Check input channels configured
TRAINING_INPUTS = [train_input, val_input, config_input]
TRAINING_INPUTS

[InputData(channel_name='train', data_source='s3://sagemaker-us-west-2-729646638167/datasets/llm-fine-tuning-estimator-sft-batch/train/dataset.json', content_type=None),
 InputData(channel_name='val', data_source='s3://sagemaker-us-west-2-729646638167/datasets/llm-fine-tuning-estimator-sft-batch/val/dataset.json', content_type=None),
 InputData(channel_name='config', data_source='s3://sagemaker-us-west-2-729646638167/datasets/llm-fine-tuning-estimator-sft-batch/config/args.yaml', content_type=None)]

***

## Queue Some Training Jobs
This section and the following are intended to be used interactively so that you can explore how to use the SageMaker Python SDK to submit jobs to your Batch queues. Let's start by selecting which queue to submit to.

### Select the Queue to Use

In [21]:
from sagemaker.train.aws_batch.training_queue import TrainingQueue

# Set the queue type to use for your job submission
SMTJ_BATCH_QUEUE = "ml-g6-12xlarge-queue"

# Construct the queue object using the SageMaker Python SDK
queue = TrainingQueue(SMTJ_BATCH_QUEUE)
print(f"Using queue: {queue.queue_name}")

Using queue: ml-g6-12xlarge-queue


### Submit your jobs
In the next cell, we are going to submit 2 Training jobs in the queue
1. LOW PRIORITY
3. MEDIUM PRIORITY

We are going to use the API `submit` to submit all the jobs

In [22]:
job_name_1 = job_name + "-low-pri"
queued_job_1 = queue.submit(
    model_trainer, TRAINING_INPUTS, job_name_1, priority=5, share_identifier="LOWPRI"
)

job_name_2 = job_name + "-mid-pri"
queued_job_2 = queue.submit(
    model_trainer, TRAINING_INPUTS, job_name_2, priority=3, share_identifier="MIDPRI"
)

## Display the Status of Running and 'In Queue' Jobs
We can use the job queue list and job queue snapshot APIs to programmaticaly view a snapshot of the jobs that the queue will run next. Keep in mind that for fair-share queues this ordering is dynamic and occassionally needs to be refreshed as new jobs are submitted to the queue or as share usage changes over time.

In [23]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

======== ml-g6-12xlarge-queue Submitted and Runnable  ===================================
 


Job : train-DeepSeek-R1-Distill-Llama-8B-sft-batch-low-pri is SUBMITTED
Job : train-DeepSeek-R1-Distill-Llama-8B-sft-batch-mid-pri is SUBMITTED





======== ml-g6-12xlarge-queue Queue Head as of 2026-07-16-09-05-23 =========================
 
    ... no jobs in queue ... 

======== ml-g6-12xlarge-queue Queue Tail =================================================== 


======== ml-g6-12xlarge-queue Starting and Running Jobs  ===================================
 


    ... no running jobs ... 




### Submit an additional job
In the next cell, we are going to submit an additional job to the queue, by using the API `submit`

In [24]:
job_name_3 = job_name + "-high-pri"
queued_job_3 = queue.submit(
    model_trainer, TRAINING_INPUTS, job_name_3, priority=1, share_identifier="HIGHPRI"
)

## Display the Status of Running and 'In Queue' Jobs
Now we are going to see another runnable job. Given that the last job has high priority, it will be run before the `MIDPRI` and `LOWPRI` jobs

In [25]:
from smtj_batch_utils.queue_utils import print_queue_state

print_queue_state(queue)

======== ml-g6-12xlarge-queue Submitted and Runnable  ===================================
 


Job : train-DeepSeek-R1-Distill-Llama-8B-sft-batch-low-pri is SUBMITTED
Job : train-DeepSeek-R1-Distill-Llama-8B-sft-batch-mid-pri is SUBMITTED


Job : train-DeepSeek-R1-Distill-Llama-8B-sft-batch-high-pri is SUBMITTED



======== ml-g6-12xlarge-queue Queue Head as of 2026-07-16-09-05-27 =========================
 
    ... no jobs in queue ... 

======== ml-g6-12xlarge-queue Queue Tail =================================================== 


======== ml-g6-12xlarge-queue Starting and Running Jobs  ===================================
 


    ... no running jobs ... 




## Cancel a Job in the Queue
This next cell shows how to cancel an in queue job.

In [26]:
runnable_jobs = queue.list_jobs(status="RUNNABLE")
if runnable_jobs:
    for job in runnable_jobs:
        job_to_cancel = job
        print(f"Cancelling job: {job_to_cancel.describe().get('jobName', '')}")
        job_to_cancel.terminate()